In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv("appointments_clean.csv")



In [ ]:
print(df.shape)
print(df.columns.tolist())

(990, 16)
['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status', 'lead_days', 'no_show']


In [ ]:
df.head()

,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
0,A0001,P509,C05,2025-01-24,2025-01-25,60,F,18.0,850,Phone,Consultation,Yes,1,No-Show,1,1
1,A0002,P046,C07,2025-05-27,2025-06-06,39,M,12.5,350,Walk-In,Follow-up,No,2,Attended,10,0
2,A0003,P009,C02,2025-03-24,2025-03-30,63,F,4.1,850,Walk-In,Consultation,No,0,Attended,6,0
3,A0004,P482,C01,2025-04-18,2025-04-22,52,F,7.5,750,App,Consultation,Yes,0,Attended,4,0
4,A0005,P251,C01,2024-11-23,2025-01-02,64,F,19.0,700,Phone,Consultation,No,0,No-Show,40,1


In [ ]:
#Task 3.1 — Features, split and baseline

In [ ]:
#converting reminder_sent to binary values through .map
df["reminder_sent"]=df["reminder_sent"].map({"Yes":1,"No":0})

In [ ]:
df["reminder_sent"].head()

,reminder_sent
0,1
1,0
2,0
3,1
4,0


In [ ]:
#creating x and y
X=df[["age", "distance_km", "fee_inr", "lead_days", "prior_no_shows", "reminder_sent","booking_channel","visit_type"]]
y=df["no_show"]

In [ ]:
#one hot encoding on booking_channel and visit_type
X = pd.get_dummies(X,columns=["booking_channel", "visit_type"],dtype=int)

In [ ]:
print(X.shape)

(990, 11)


In [ ]:
#splitting data in training and testing
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (792, 11)
X_test: (198, 11)
y_train: (792,)
y_test: (198,)


In [ ]:
#Number  of no show
print("Number  of no show",y_test.sum())

Number  of no show 49


In [ ]:
#attended appointments
print("attended appointments",(y_test==0).sum())

attended appointments 149


In [ ]:
#baseline accuracy
print("baseline accuracy",(y_test==0).mean())

baseline accuracy 0.7525252525252525


In [ ]:
#Task 3.2 — Logistic regression

In [ ]:
#scaling
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
scaler.fit(X_train) #learning mu and std from training data only
X_train_s =scaler.transform(X_train) #scales the training data
X_test_s=scaler.transform(X_test) #scales the testing data

In [ ]:
print(X_train_s.shape)
print(X_test_s.shape)

(792, 11)
(198, 11)


In [ ]:
#logistic regression
from sklearn.metrics import recall_score,accuracy_score,precision_score,confusion_matrix, f1_score,roc_auc_score
lr_model=LogisticRegression(max_iter=1000)
lr_model.fit(X_train_s,y_train)
y_pred=lr_model.predict(X_test_s)


In [ ]:
#calculating accuracy precision, recall, F1

accuracy=accuracy_score(y_test,y_pred)
precision=precision_score(y_test,y_pred)
recall=recall_score(y_test,y_pred)
f1=f1_score(y_test,y_pred)
print("accuracy=",accuracy)
print("precision=",precision)
print("recall=",recall)
print("f1=",f1)

accuracy= 0.7878787878787878
precision= 0.6296296296296297
recall= 0.3469387755102041
f1= 0.4473684210526316


In [ ]:
#the confusion matrix
cm=confusion_matrix(y_test,y_pred)
print("confusion_matrix=",cm)

confusion_matrix= [[139  10]
 [ 32  17]]


In [ ]:
# ROC-AUC
y_prob = lr_model.predict_proba(X_test_s)[:, 1]
roc_auc=roc_auc_score(y_test,y_prob)

In [ ]:
#Logistic Regression coefficients.
for feature, coef in zip(X.columns, lr_model.coef_[0]):
    print(feature, coef)
coefficient=lr_model.coef_[0]
np.exp(coefficient)

age -0.2538801442716842
distance_km 0.25344770956317536
fee_inr 0.4527614630856424
lead_days 0.6144707304969083
prior_no_shows 0.6496116219621334
reminder_sent -0.5806686033659922
booking_channel_App -0.04680357040738989
booking_channel_Phone 0.2158813749282703
booking_channel_Walk-In -0.18636416836300682
visit_type_Consultation 0.01565801179835534
visit_type_Follow-up -0.015658011798355072


array([0.77578478, 1.28846   , 1.57264901, 1.84867789, 1.91479702,
       0.55952414, 0.95427483, 1.24095516, 0.82997129, 1.01578124,
       0.98446394])

In [ ]:
reminder_index = X.columns.get_loc("reminder_sent")

reminder_coef = lr_model.coef_[0][reminder_index]

reminder_sd = scaler.scale_[reminder_index]

actual_effect = np.exp(reminder_coef / reminder_sd)

print(actual_effect)

0.3093980903124433


In [ ]:
#Task 3.3 — Decision tree and overfitting

In [ ]:
from sklearn.tree import DecisionTreeClassifier
dt=DecisionTreeClassifier(random_state=42)
dt.fit(X_train,y_train)   # scaling not required for dt


DecisionTreeClassifier(random_state=42)

In [ ]:
#calculating both test and train accuracy
print("Unrestricted tree")
y_dt_predict=dt.predict(X_test)
test_accuracy=accuracy_score(y_test,y_dt_predict)
print("test_accuracy",test_accuracy)

X_dt_predict=dt.predict(X_train)
train_accuracy=accuracy_score(y_train,X_dt_predict)
print("train_accuracy",train_accuracy)

Unrestricted tree
test_accuracy 0.6717171717171717
train_accuracy 1.0


In [ ]:
#dt with max depth 3
dt_3=DecisionTreeClassifier(random_state=42,max_depth=3)
dt_3.fit(X_train,y_train)


DecisionTreeClassifier(max_depth=3, random_state=42)

In [ ]:
print("tree with max _depth=3")
dt_3_predict=dt_3.predict(X_test)
test_accuracy_3=accuracy_score(y_test,dt_3_predict)
print("test accuracy",test_accuracy_3)
dt_3_predict_train=dt_3.predict(X_train)
train_accuracy_3=accuracy_score(y_train,dt_3_predict_train)
print("train accuracy",train_accuracy_3)

tree with max _depth=3
test accuracy 0.7777777777777778
train accuracy 0.821969696969697


without the max_depth the model **overfits as training accuracy =100% nd testing accuracy =67.17%**....Hence after using max_depth=3 **training accuracy decreases to 82% but the testing accuracy has increased to 77%** hence better results than before and **reduces overfitting**

In [ ]:
#Task 3.4 — Random forest and a comparison table

In [ ]:
from sklearn.ensemble import RandomForestClassifier
RF=RandomForestClassifier(n_estimators=100, random_state=42)
RF.fit(X_train,y_train)
y_pred_RF=RF.predict(X_test)

In [ ]:
#calculating metrics- accuracy, recall, aur_roc
accuracy_rf=accuracy_score(y_test,y_pred_RF)
print("accuracy=",accuracy_rf)

recall_rf=recall_score(y_test,y_pred_RF)
print("recall_rf=",recall_rf)

y_prob_rf=RF.predict_proba(X_test)[:,1]
roc_auc_rf = roc_auc_score(y_test, y_prob_rf)
print("ROC_AUC=",roc_auc_rf)

accuracy= 0.7474747474747475
recall_rf= 0.30612244897959184
ROC_AUC= 0.7057252431173813


In [ ]:
#feature importance
importance = pd.Series(
    RF.feature_importances_,
    index=X.columns
)

importance.sort_values(ascending=False).head(3)


,0
lead_days,0.219905
age,0.196788
distance_km,0.194739


In [ ]:
#recall and roc auc for dt with max depth =3
print("Decision Tree with max depth= 3")
recall_dt3 = recall_score(y_test, dt_3_predict)
print("recall=",recall_dt3)
y_prob_dt3=dt_3.predict_proba(X_test)[:,1]
roc_auc_dt3=roc_auc_score(y_test,y_prob_dt3)
print("AUC ROC=",roc_auc_dt3)

Decision Tree with max depth= 3
recall= 0.22448979591836735
AUC ROC= 0.6996986714148747


In [ ]:
#table
pd.DataFrame(
    {
     "Model":["Logistic_Regression", "Decision_Tree_depth3" ,"Random_Forest"],
     "Accuracy":[accuracy,test_accuracy_3,accuracy_rf],
     "Recall":[recall,recall_dt3,recall_rf],
     "ROC-AUC":[roc_auc,roc_auc_dt3,roc_auc_rf]
    }
             )

,Model,Accuracy,Recall,ROC-AUC
0,Logistic_Regression,0.787879,0.346939,0.788933
1,Decision_Tree_depth3,0.777778,0.224490,0.699699
2,Random_Forest,0.747475,0.306122,0.705725


In [ ]:
#Task 3.5 — Tune, choose the model and the threshold, then test once


In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
#grid search on RFC
grid_search=GridSearchCV(RandomForestClassifier(random_state=42),{
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 8]
},cv=5,scoring="roc_auc")
grid_search.fit(X_train,y_train)

GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42),
             param_grid={'max_depth': [3, 5, 8], 'n_estimators': [100, 200]},
             scoring='roc_auc')

In [ ]:
print("best parameter=",grid_search.best_params_)
print("best score=",grid_search.best_score_)

best parameter= {'max_depth': 3, 'n_estimators': 200}
best score= 0.7673242251739405


In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_val_predict
from sklearn.pipeline import Pipeline

In [ ]:
#creating pipeline fir logistic regression
lr_pipe=Pipeline([
    ("scaler",StandardScaler()),
    ("model",LogisticRegression(max_iter=1000))
     ])


In [ ]:
cv_auc=cross_val_score(lr_pipe,X_train,y_train,cv=5,scoring="roc_auc").mean()
dt_auc=cross_val_score(dt_3,X_train,y_train,cv=5,scoring="roc_auc").mean()
rf_auc=cross_val_score(RF,X_train,y_train,cv=5,scoring="roc_auc").mean()

print("Logistic Regression average ROC",cv_auc)
print("Decision Tree average ROC",dt_auc)
print("RandomForestClassifier average ROC",rf_auc)

Logistic Regression average ROC 0.7737139899948249
Decision Tree average ROC 0.6784494968661952
RandomForestClassifier average ROC 0.7466024782933701


In [ ]:
oof = cross_val_predict(lr_pipe, X_train, y_train, cv=5, method="predict_proba")[:, 1]
oof.shape

(792,)

In [ ]:
pred_50 =(oof >=0.5).astype(int)
pred_35=(oof >=0.35).astype(int)

In [ ]:
precision_5=precision_score(y_train,pred_50)
recall_5=recall_score(y_train,pred_50)
f1_5=f1_score(y_train,pred_50)

print("Metrics for thresold 0.5")
print("precision",precision_5)
print("recall",recall_5)
print("f1",f1_5)
print("number flagged",pred_50.sum())

Metrics for thresold 0.5
precision 0.5753424657534246
recall 0.2485207100591716
f1 0.34710743801652894
number flagged 73


In [ ]:
precision_035=precision_score(y_train,pred_35)
recall_035=recall_score(y_train,pred_35)
f1_035=f1_score(y_train,pred_35)

print("Metrics for thresold 0.35")
print("precision",precision_035)
print("recall",recall_035)
print("f1",f1_035)
print("number flagged",pred_35.sum())

Metrics for thresold 0.35
precision 0.512987012987013
recall 0.46745562130177515
f1 0.4891640866873065
number flagged 154


In [ ]:
lr_pipe.fit(X_train,y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('model', LogisticRegression(max_iter=1000))])

In [ ]:
probs=lr_pipe.predict_proba(X_test)[:,1]
final_pred=(probs>0.35).astype(int)

In [ ]:
precision=precision_score(y_test,final_pred)
recall=recall_score(y_test,final_pred)
f1=f1_score(y_test,final_pred)
roc_auc = roc_auc_score(y_test, probs) #require probabilty for roc_auc


print("Metrics for thresold 0.35")
print("precision",precision)
print("recall",recall)
print("number flagged",final_pred.sum())
print("f1",f1)
print("ROC-AUC:", roc_auc)

Metrics for thresold 0.35
precision 0.5714285714285714
recall 0.5714285714285714
number flagged 49
f1 0.5714285714285714
ROC-AUC: 0.788933022873579


In [ ]:
test_ids = df.loc[X_test.index, "appointment_id"]
test_ids.head()

,appointment_id
213,A0217
331,A0336
501,A0508
309,A0314
88,A0089


In [ ]:
#combining the required columns and saving it
test_risk_scores = pd.DataFrame({
    "appointment_id": test_ids.to_numpy(),
    "no_show": y_test.to_numpy(),
    "risk_prob": probs,
    "flag": final_pred
})


In [ ]:
print(test_risk_scores.head())
print(test_risk_scores.shape)

  appointment_id  no_show  risk_prob  flag
0          A0217        0   0.021565     0
1          A0336        0   0.150917     0
2          A0508        0   0.518692     1
3          A0314        0   0.191300     0
4          A0089        0   0.167718     0
(198, 4)


In [ ]:
#saving the file
test_risk_scores.to_csv("test_risk_scores.csv", index=False)